# 04 — Unsupervised niche clustering + biology

Step 6 + 6.5 + 7. Trains a Graph Auto-Encoder (or DGI) on every cell's
niche, projects all 259k niches into a 64-dim embedding space, runs Leiden
to discover niche-clusters, then enriches each cluster with:

- pathway enrichment on its top marker genes
- ligand-receptor interactions across its niches
- example niche visualisations with cell-type colouring

Adversarial debiasing against `patient_id` pushes the encoder toward
patient-invariant representations.

Outputs are persisted as `ecof_annotated.h5ad` — a copy of the input
AnnData with `obs['ecof_niche_cluster']`, `obsm['ecof_niche_embedding']`
and `obsm['ecof_niche_umap']` attached.

In [ ]:
import sys; from pathlib import Path
REPO = Path.cwd().parent.parent
if str(REPO / 'src') not in sys.path: sys.path.insert(0, str(REPO / 'src'))

from ecofoundation.config.loader import load_config
from ecofoundation.pipelines.unsup_cluster import run_unsup_clustering_pipeline

## 1. Run the pipeline

Defaults: GAE, 30% subsample for training, adversarial debiasing on
`patient_id`, AnnData export enabled.

Switch to DGI by setting `cfg.unsup.model.architecture = 'dgi'`.

In [ ]:
cfg = load_config(REPO / 'configs/unsup_clustering.yaml')
# cfg.unsup.model.architecture = 'dgi'   # alternative
# cfg.unsup.adversarial.lambda_max = 0.3  # gentler debias
folder = run_unsup_clustering_pipeline(cfg)
print('Report :', folder.report_path)
print('AnnData:', folder.artifacts_dir / 'ecof_annotated.h5ad')

## 2. Load the annotated AnnData

Standard scanpy-style usage from here on.

In [ ]:
import anndata as ad
a = ad.read_h5ad(folder.artifacts_dir / 'ecof_annotated.h5ad', backed='r')
print(a)
print('Niche-cluster counts:')
print(a.obs['ecof_niche_cluster'].value_counts().head())

## 3. Cluster-biology artefacts

The Step-7 deep-dive writes three parquet tables.

In [ ]:
import pandas as pd
summary = pd.read_parquet(folder.artifacts_dir / 'niche_cluster_summary.parquet')
markers = pd.read_parquet(folder.artifacts_dir / 'niche_cluster_markers.parquet')
summary

In [ ]:
markers[markers['rank'] <= 3].head(20)

In [ ]:
pathways_path = folder.artifacts_dir / 'cluster_pathways.parquet'
if pathways_path.exists():
    pathways = pd.read_parquet(pathways_path)
    display(pathways.groupby('cluster').head(2)[['cluster','term','adjusted_p_value','combined_score']])

In [ ]:
lr_path = folder.artifacts_dir / 'cluster_lr_interactions.parquet'
if lr_path.exists():
    lr = pd.read_parquet(lr_path)
    display(lr.groupby('cluster').head(3)[['cluster','ct_pair_a','ct_pair_b','ligand','receptor','score_sum']])

## 4. Plots, on demand

All factory functions live in `ecofoundation.reporting.plots`; pass the
cluster_stats / biology objects directly.

In [ ]:
from ecofoundation.reporting.plots import (
    niche_cluster_composition_bar, niche_cluster_marker_heatmap,
    cluster_pathway_dotplot, cluster_lr_heatmap,
)
from ecofoundation.niches.cluster_characterization import NicheClusterStats
from ecofoundation.niches.cluster_biology import ClusterBiology

comp = pd.read_parquet(folder.artifacts_dir / 'niche_cluster_composition.parquet')
stats_lite = NicheClusterStats(
    composition=comp,
    size_distribution=pd.DataFrame(),
    sample_distribution=pd.DataFrame(),
    markers=markers,
    cluster_summary=summary,
)
fig = niche_cluster_composition_bar(stats_lite)
fig

In [ ]:
fig = niche_cluster_marker_heatmap(stats_lite, n_top=5)
fig

In [ ]:
if pathways_path.exists() and lr_path.exists():
    biology = ClusterBiology(
        pathways=pd.read_parquet(pathways_path),
        lr_interactions=pd.read_parquet(lr_path),
        example_niches={},
    )
    display(cluster_pathway_dotplot(biology))
    display(cluster_lr_heatmap(biology))

## 5. Cell-level niche cluster on the spatial map

`scanpy.pl.spatial` works directly on the exported AnnData.

In [ ]:
# Load fully into memory for plotting
import anndata as ad
a_full = ad.read_h5ad(folder.artifacts_dir / 'ecof_annotated.h5ad')
# Drop cells without a niche assignment for cleaner plots
sub = a_full[a_full.obs['ecof_niche_cluster'].astype(str) != 'unassigned']
print(sub.obs['ecof_niche_cluster'].value_counts().head())

# Optional: render via the project's matplotlib helpers
from ecofoundation.reporting.plots import niche_cluster_spatial
import numpy as np
fig = niche_cluster_spatial(
    sub,
    sample_key='samples',
    spatial_key='spatial',
    cluster_per_cell=np.array(sub.obs['ecof_niche_cluster'].astype(str)),
    max_points_per_sample=4000,
)
fig